In [29]:
machakos_raw =  ["48", "35", "88", "N/A", "140", " 20 ",
                 "9", "12", "", "45", "182", "96"]

months =  ["January", "February", "March", "April",
          "May", "June", "July", "August",
          "September", "October", "November", "December"]

def clean_machakos(data, months):
    valid_values = []
    cleaned_values = []
    imputed_months = []

    for value in data:
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(None)

    for value in cleaned_values:
        if value is not None:
            valid_values.append(value)

    average_rainfall = round(sum(valid_values) / len(valid_values))

    for i in range(len(cleaned_values)):
        if cleaned_values[i] is None:
            cleaned_values[i] = average_rainfall
            imputed_months.append(months[i])

    print("Average rainfall used for missing values:", average_rainfall, "mm")
    print("Months with imputed values:", imputed_months)
    print("Cleaned Machakos rainfall data:", cleaned_values)

    return cleaned_values

Machakos_clean = clean_machakos(machakos_raw, months)

Average rainfall used for missing values: 68 mm
Months with imputed values: ['April', 'September']
Cleaned Machakos rainfall data: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [30]:
import numpy as np

#List of counties
counties = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]   # Mombasa
]

# Initialize clean_rows with the 4 existing county lists
clean_rows = list(counties)
clean_rows.append(Machakos_clean)

#combine all 5 counties into a Numpy array
R = np.array(clean_rows)

print("Combined rainfall array:")
print(R)

print("\nArray shape:", R.shape)
print("Array data type:", R.dtype)

Combined rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]

Array shape: (5, 12)
Array data type: int64


In [31]:
# Define the names for all 5 counties in the array
county_names = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

# Calculate the annual total rainfall for each county
annual_rainfall = np.sum(R, axis=1)
print("Annual Total Rainfall per County:")

for i in range(len(county_names)):
    print(county_names[i], ":", annual_rainfall[i])

# Calculate the average rainfall per month across all counties
average_monthly_rainfall = np.mean(R, axis=0)

print("\nAverage Rainfall per Month Across All Counties:")

for i in range(12):
    print(months[i], ":", average_monthly_rainfall[i])

Annual Total Rainfall per County:
Nairobi : 1045
Nakuru : 916
Kisumu : 1371
Mombasa : 1075
Machakos : 811

Average Rainfall per Month Across All Counties:
January : 47.0
February : 47.6
March : 91.0
April : 164.4
May : 171.2
June : 67.6
July : 56.6
August : 58.2
September : 64.2
October : 67.8
November : 127.4
December : 80.6


In [32]:
wettest_month_index = np.argmax(R, axis=1)
print("Wettest Month for Each County:")

for county, index in zip(county_names, wettest_month_index):
    print(county, ":", months[index])

Wettest Month for Each County:
Nairobi : April
Nakuru : April
Kisumu : April
Mombasa : May
Machakos : November


In [33]:
# dry months with rainfal below 50mm
dry_months = R < 50

#number of dry months in each county
dry_month_count = np.sum(dry_months, axis=1)
print("Number of Dry Months per County (below 50 mm):")

for county, count in zip(county_names, dry_month_count):
    print(county, ":", count, "dry months")


Number of Dry Months per County (below 50 mm):
Nairobi : 4 dry months
Nakuru : 3 dry months
Kisumu : 0 dry months
Mombasa : 2 dry months
Machakos : 6 dry months


In [34]:
# the recommended planinting window
consecutive_rainfall = R[:, :-1] + R[:, 1:]
best_planting_index = np.argmax(consecutive_rainfall, axis=1)
print("Recommended Planting Window per County:")

for county, index in zip(county_names, best_planting_index):
    print(
        county, ":",
        months[index], "and", months[index + 1],
        "-", consecutive_rainfall[county_names.index(county), index], "mm"
    )

Recommended Planting Window per County:
Nairobi : April and May - 395 mm
Nakuru : April and May - 265 mm
Kisumu : April and May - 420 mm
Mombasa : April and May - 390 mm
Machakos : November and December - 278 mm


In [35]:
import csv

with open("rainfall_summary.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    planting_windows = [
        months[i] + " - " + months[i + 1]
        for i in best_planting_index
    ]

    writer.writerows(zip(
        county_names,
        annual_rainfall,
        [months[i] for i in wettest_month_index],
        dry_month_count,
        planting_windows
    ))

print("rainfall_summary.csv created successfully!")

with open("rainfall_summary.csv", "r") as file:
    print(file.read())

from google.colab import files
files.download("rainfall_summary.csv")

rainfall_summary.csv created successfully!
county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April - May
Nakuru,916,April,3,April - May
Kisumu,1371,April,0,April - May
Mombasa,1075,May,2,April - May
Machakos,811,November,6,November - December



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Planting Advisory

The NGO should advise farmers to plan their planting around the months that receive the most rainfall. April is the wettest month for most of the counties, so farmers should prepare for planting during this period. Counties with more dry months may need extra water support. The NGO should also help farmers prepare for months with low rainfall. However, the Machakos rainfall data needed cleaning because some values were missing or incorrectly recorded. This means the Machakos results may not be fully accurate.